# Freesound SVM Classification (Core)

This notebook demonstrates a simple workflow for sound classification using SVMs and Freesound data.

The workflow is split into three clear steps:

1. **Download and prepare data**
2. **Train the SVM model**
3. **Classify a query sound**

Each step is in a separate code cell for clarity.

In [19]:
import os
import json
from pathlib import Path
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import freesound as fs

# --- Descriptor selection ---
descriptors = [
    "lowlevel.spectral_centroid.mean",
    "lowlevel.spectral_contrast.mean",
    "lowlevel.dissonance.mean",
    "lowlevel.hfc.mean",
    "lowlevel.mfcc.mean",
    "sfx.logattacktime.mean",
    "sfx.inharmonicity.mean",
]
descriptor_mapping = {
    0: "lowlevel.spectral_centroid.mean",
    1: "lowlevel.dissonance.mean",
    2: "lowlevel.hfc.mean",
    3: "sfx.logattacktime.mean",
    4: "sfx.inharmonicity.mean",
    5: "lowlevel.spectral_contrast.mean.0",
    6: "lowlevel.spectral_contrast.mean.1",
    7: "lowlevel.spectral_contrast.mean.2",
    8: "lowlevel.spectral_contrast.mean.3",
    9: "lowlevel.spectral_contrast.mean.4",
    10: "lowlevel.spectral_contrast.mean.5",
    11: "lowlevel.mfcc.mean.0",
    12: "lowlevel.mfcc.mean.1",
    13: "lowlevel.mfcc.mean.2",
    14: "lowlevel.mfcc.mean.3",
    15: "lowlevel.mfcc.mean.4",
    16: "lowlevel.mfcc.mean.5",
}

# --- Simple helpers ---
def download_sounds(query, tag, duration, api_key, out_dir, n_results=10):
    client = fs.FreesoundClient()
    client.set_token(api_key, "token")
    # Build filter string correctly
    filter_parts = []
    if tag and tag.strip():
        filter_parts.append(f"tag:{tag}")
    filter_parts.append(f"duration:[{duration[0]} TO {duration[1]}]")
    filter_str = " ".join(filter_parts)
    results = client.text_search(
        query=query,
        filter=filter_str,
        page_size=n_results,
        fields="id,name,previews,analysis",
        descriptors=','.join(descriptors),
        normalized=1
    )
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for sound in results:
        sdir = out_dir / str(sound.id)
        sdir.mkdir(exist_ok=True)
        # Save descriptor JSON only if analysis is available
        if sound.analysis is not None:
            try:
                desc = {}
                for d in descriptors:
                    value = sound.analysis
                    for part in d.split('.'):
                        value = getattr(value, part, None)
                        if value is None:
                            break
                    desc[d] = [value]
                with open(sdir / "desc.json", "w") as f:
                    json.dump(desc, f)
            except Exception as e:
                print(f"Could not extract descriptors for sound {sound.id}: {e}")
        else:
            print(f"No analysis data for sound {sound.id}, skipping.")

def load_features_labels(data_dir):
    X, y = [], []
    for class_dir in Path(data_dir).iterdir():
        if class_dir.is_dir():
            for sound_dir in class_dir.iterdir():
                desc_file = sound_dir / "desc.json"
                if desc_file.exists():
                    with open(desc_file) as f:
                        d = json.load(f)
                    # Example: use centroid and mean of mfcc
                    X.append([
                        d["lowlevel.spectral_centroid.mean"][0],
                        np.mean(d["lowlevel.mfcc.mean"][0])
                    ])
                    y.append(class_dir.name)
    return np.array(X), np.array(y)


## 2. Download and Prepare Data

Set the parameters for your training and query data search here, then run the cell to download and prepare the data.

In [ ]:
# --- Set parameters for training and query data ---
load_dotenv()
api_key = os.environ.get("FREESOUND_API_KEY")

# Training set parameters (edit these to control the search)
train_classes = [
    {"name": "violin", "tag": "pizzicato", "n": 5},
    {"name": "naobo", "tag": "", "n": 5},
    {"name": "trumpet", "tag": "single-note", "n": 5},
]
train_duration = (0, 3)
train_dir = "svm_train_sounds"

# Query set parameters
query_name = "viola"
query_tag = "pizzicato"
query_n = 1
query_duration = (0, 3)
query_dir = "svm_query_sound"

# --- Download training data ---
for c in train_classes:
    download_sounds(c["name"], c["tag"], train_duration, api_key, Path(train_dir) / c["name"], n_results=c["n"] )

# --- Download query data ---
download_sounds(query_name, query_tag, query_duration, api_key, Path(query_dir) / query_name, n_results=query_n)

# --- Prepare features and labels ---
X, y = load_features_labels(train_dir)
if len(X) == 0 or len(y) == 0:
    raise ValueError("No features or labels found. Check that sounds were downloaded and descriptors extracted correctly.")
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42)


No analysis data for sound 374529, skipping.
No analysis data for sound 374528, skipping.
No analysis data for sound 374527, skipping.
No analysis data for sound 374526, skipping.
No analysis data for sound 374525, skipping.
No analysis data for sound 222298, skipping.
No analysis data for sound 222297, skipping.
No analysis data for sound 222296, skipping.
No analysis data for sound 222295, skipping.
No analysis data for sound 222294, skipping.
No analysis data for sound 357559, skipping.
No analysis data for sound 357558, skipping.
No analysis data for sound 357557, skipping.
No analysis data for sound 357556, skipping.
No analysis data for sound 357479, skipping.
No analysis data for sound 374390, skipping.


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## 3. Classify a Query Sound

This cell loads a query sound, extracts its features, and predicts its class using the trained SVM model.

In [ ]:
# Classify a query sound
query_desc = Path(query_dir) / "viola" / "1" / "desc.json"
with open(query_desc) as f:
    d = json.load(f)
query_feat = np.array([[d["lowlevel.spectral_centroid.mean"][0], np.mean(d["lowlevel.mfcc.mean"][0])]])
pred = model.predict(query_feat)[0]
proba = model.predict_proba(query_feat).max()
print(f"Predicted class for query: {pred} (probability {proba:.2f})")

## Evaluation

Evaluate the trained SVM on the training and test splits using accuracy, precision, recall, F1-score, and the confusion matrix.

In [ ]:
# Evaluate trained model on training and test sets
train_pred = model.predict(scaled_train_features)
test_pred = model.predict(scaled_test_features)

print(f'Train accuracy: {accuracy_score(train_labels, train_pred) * 100:.1f}%')
print(f'Test accuracy: {accuracy_score(test_labels, test_pred) * 100:.1f}%')
print(f'Test precision: {precision_score(test_labels, test_pred, average="weighted", zero_division=0) * 100:.1f}%')
print(f'Test recall: {recall_score(test_labels, test_pred, average="weighted", zero_division=0) * 100:.1f}%')
print(f'Test F1: {f1_score(test_labels, test_pred, average="weighted", zero_division=0) * 100:.1f}%')

classes = sorted(np.unique(test_labels))
print('Confusion matrix (test):')
print(confusion_matrix(test_labels, test_pred, labels=classes))

## Query prediction and result export

Download the query sound if needed, classify it with the trained model, and save the result bundle for the visualization notebook.

In [ ]:
# Resolve or download query descriptor, then predict and save results bundle
query_root = Path('svm_query_sound') / query_sound_name
query_candidates = sorted(query_root.rglob('*.json'))

if not query_candidates:
    print(f"No local query descriptor found for '{query_sound_name}'. Searching in Freesound...")
    load_dotenv()
    api_key = os.environ.get('FREESOUND_API_KEY')
    if not api_key:
        raise EnvironmentError(
            'FREESOUND_API_KEY not set. '
            'Create a .env file with FREESOUND_API_KEY=your_key (see .env.example).'
        )

    download_sounds_freesound(
        queryText=query_sound_name,
        tag=query_tag,
        duration=query_duration,
        API_Key=api_key,
        outputDir='svm_query_sound',
        topNResults=query_top_n_results,
    )
    query_candidates = sorted(query_root.rglob('*.json'))

if not query_candidates:
    available = []
    base = Path('svm_query_sound')
    if base.exists():
        available = sorted([p.name for p in base.iterdir() if p.is_dir()])
    raise FileNotFoundError(
        f"No query descriptor found for '{query_sound_name}'.\n"
        f"Expected under: {query_root}\n"
        f"Available query folders: {available}\n"
        f"Freesound search returned no usable descriptor files."
    )

query_file = str(query_candidates[0])
print(f'Using query descriptor: {query_file}')

with open(query_file, 'r', encoding='utf-8') as file_obj:
    query_dict = json.load(file_obj)
query_features = convFtrDict2List(query_dict)[selected_descriptors].astype(float)
scaled_query_features = (query_features - mean.ravel()) / std.ravel()

predicted_label = model.predict([scaled_query_features])[0]
probabilities = model.predict_proba([scaled_query_features])[0]
predicted_probability = float(dict(zip(model.classes_, probabilities))[predicted_label])
query_label = _infer_query_class(query_file)

combined_scaled_features = np.vstack((scaled_train_features, scaled_test_features))
combined_labels = np.concatenate((train_labels, test_labels))
combined_sound_ids = [sound_ids[i] for i in train_indices] + [sound_ids[i] for i in test_indices]
test_mask = np.concatenate((np.zeros(len(train_features), dtype=bool), np.ones(len(test_features), dtype=bool)))

bundle = {
    'combined_scaled_features': combined_scaled_features,
    'combined_labels': combined_labels,
    'combined_sound_ids': combined_sound_ids,
    'test_mask': test_mask,
    'support_indices': model.support_,
    'scaled_query_features': scaled_query_features,
    'query_label': query_label,
    'predicted_label': predicted_label,
    'predicted_probability': predicted_probability,
    'kernel': kernel,
    'C': C,
    'gamma': gamma,
}

results_path = Path(results_file)
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, 'wb') as file_obj:
    pickle.dump(bundle, file_obj)

print(f'Predicted query class: {predicted_label} ({predicted_probability * 100:.1f}%)')
print(f'Saved core results to: {results_path}')